# LLM file labeling on Colab GPU

Runs `model.generate_training_data` against an open model served by vLLM on the Colab GPU.

**Runtime → Change runtime type → GPU** (T4 works; L4 / A100 can serve a larger model).

Outputs (label cache, training JSONL) go to Google Drive, so a disconnected session resumes from the cache.
To run the gold-set evaluation, upload `model/tmp/eval_llm_labels.py` (gitignored) to the Drive folder `WORK` first.

## 1. Drive, code, source repos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/chitlist_labeling'   # cache + outputs live here
import os
os.makedirs(WORK, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Project code (push local changes first — this clones origin/master)
!rm -rf /content/Chained && git clone -q https://github.com/qpdbcoocdbqp/Chained.git /content/Chained
%cd /content/Chained
!mkdir -p model/tmp
# Gold-set eval script is gitignored; copy it from Drive if uploaded
!test -f {WORK}/eval_llm_labels.py && cp {WORK}/eval_llm_labels.py model/tmp/ && echo "eval script copied" || echo "no eval script in WORK — skipping eval"

In [ ]:
# Training repos, pinned to the same commits as the local source/ checkout
# (the gold sets in eval_llm_labels.py and model/tmp/gold/ refer to files at these commits)
SOURCE_REPOS = {
    'llama.cpp': ('https://github.com/ggml-org/llama.cpp.git', '07d937828636e305bc0cfe738b288f9ab05ff748'),
    'milvus':    ('https://github.com/milvus-io/milvus.git',   '6c8462cefb59b81b3332c622f80e384c3a6d6361'),
    # tags: go-redis v9.23.0, httpx 0.28.1, curl 8.23.0, vite v8.3.4, ollama v0.40.6
    'go-redis':  ('https://github.com/redis/go-redis.git',     '6f2b2639e563b11709e06f42bc0addbfb5a1abb6'),
    'httpx':     ('https://github.com/encode/httpx.git',       '26d48e0634e6ee9cdc0533996db289ce4b430177'),
    'curl':      ('https://github.com/curl/curl.git',          '3163ec065643ba20709f190111f42c3e1fe95273'),
    'vite':      ('https://github.com/vitejs/vite.git',        'b6c20f6a8126d8561891e5fb9368e8fc9da37538'),
    'ollama':    ('https://github.com/ollama/ollama.git',      '2476d5ead197abe04629651028f83d48dc75d2b2'),
}
import subprocess
for name, (url, sha) in SOURCE_REPOS.items():
    d = f'/content/source/{name}'
    if not os.path.isdir(d):
        !git init -q {d} && git -C {d} remote add origin {url}
        !git -C {d} fetch -q --depth 1 origin {sha} && git -C {d} checkout -q FETCH_HEAD
    n_files = len(subprocess.run(['git', '-C', d, 'ls-files'], capture_output=True, text=True).stdout.splitlines())
    print(f'{name}: {n_files} files')

## 2. Start vLLM

`LABEL_MODEL` picks the labeling model:

| `LABEL_MODEL` | Model | Use |
|---|---|---|
| `'auto'` (default) | Qwen2.5-7B / 14B / 32B-Instruct-AWQ by GPU memory | training data |
| `'fast'` | Qwen2.5-3B-Instruct-AWQ | faster labeling: on a T4 about 36 batches fit in the KV cache at once against about 16 for the 7B, with half the weights to read per token |
| `'fastest'` | Qwen2.5-1.5B-Instruct-AWQ | quick experiments; likely below the quality bar |
| `'qwen3.5-4b'` | Qwen3.5-4B + MTP | L4 or better: about 32 batches at once on an L4, only about 9 on a T4 (9 GB of bf16 weights) |
| `'qwen3.5-2b'` | Qwen3.5-2B + MTP | fast on any GPU: about 68 batches at once on a T4, 125 on an L4 |
| `'qwen3.5-0.8b'` | Qwen3.5-0.8B + MTP | quick experiments; likely below the quality bar |
| a Hugging Face id | that model | anything else vLLM serves |

Qwen3.5 is a hybrid model: 3 of every 4 layers use linear attention with a fixed-size state, so a batch needs far less KV cache than with Qwen2.5. It is served text only (`--language-model-only`), with its multi-token-prediction head as the speculative drafter (`qwen3_next_mtp`, 2 tokens), in bf16 where the GPU supports it, and with thinking switched off per request (`LOCAL_LLM_EXTRA_BODY`), since it thinks by default. It needs vLLM 0.17 or later. On a T4 it runs in fp16 and has not been tried here; check `/content/vllm.log` if the server does not come up. vLLM logs how many drafted tokens are accepted; MTP gains the most when few batches are in flight.

`SERVED_NAME` is part of the label cache key, so labels from different models never mix: a smaller model relabels **every** repo, including those already in the Drive cache, and writes its own `llm_training_data_<model>.jsonl`.

Run section 3 before labeling everything. On that gold set the cached Qwen2.5-7B v9 labels score accuracy 0.96, precision 0.91, recall 1.00; a model clearly below that makes worse training data, however fast.

Qwen3-4B is not offered: with 8 KV heads only about 9 batches fit in a T4's KV cache, so it is slower than the 7B here, and it thinks before answering unless told not to.

In [ ]:
# vLLM in Colab's own Python, with pip. It replaces torch, which leaves the
# preinstalled torchaudio built against another torch / CUDA and breaks
# `import transformers`; nothing here uses torchaudio, so it goes.
# -U: Colab may ship an older vLLM, and Qwen3.5 needs 0.17 or later.
!pip install -q -U vllm python-dotenv pyyaml openai
!pip uninstall -y -q torchaudio
!sed -n '15,30p' /usr/local/lib/python3.12/dist-packages/transformers/loss/loss_rnnt.py
!python -c "import transformers.loss.loss_utils; print('transformers OK')"
!vllm --version
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'gpu ok:', torch.cuda.is_available())"

In [ ]:
# 'auto' | 'fast' | 'fastest' | 'qwen3.5-4b' | 'qwen3.5-2b' | 'qwen3.5-0.8b'
# | a Hugging Face model id (see the table above)
LABEL_MODEL = 'auto'

import json
# nvidia-smi rather than torch: pip just replaced torch in this session
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=memory.total,compute_cap',
                               '--format=csv,noheader,nounits'], text=True).splitlines()[0].split(',')
gpu_gb, compute_cap = int(gpu[0]) / 1024, float(gpu[1])
NAMED_MODELS = {'fast': 'Qwen/Qwen2.5-3B-Instruct-AWQ', 'fastest': 'Qwen/Qwen2.5-1.5B-Instruct-AWQ',
                'qwen3.5-4b': 'Qwen/Qwen3.5-4B', 'qwen3.5-2b': 'Qwen/Qwen3.5-2B',
                'qwen3.5-0.8b': 'Qwen/Qwen3.5-0.8B'}
if LABEL_MODEL in NAMED_MODELS: MODEL = NAMED_MODELS[LABEL_MODEL]
elif LABEL_MODEL != 'auto':     MODEL = LABEL_MODEL
elif gpu_gb < 20:               MODEL = 'Qwen/Qwen2.5-7B-Instruct-AWQ'    # T4 16GB
elif gpu_gb < 40:               MODEL = 'Qwen/Qwen2.5-14B-Instruct-AWQ'   # L4 24GB
else:                           MODEL = 'Qwen/Qwen2.5-32B-Instruct-AWQ'   # A100 40/80GB
SERVED_NAME = MODEL.split('/')[-1].lower()

# Qwen3.5: text only, its multi-token-prediction head as the speculative
# drafter, thinking off (it thinks by default), bf16 where the GPU has it.
# AWQ models need fp16, and a T4 has no bf16.
QWEN35 = MODEL.startswith('Qwen/Qwen3.5-')
MTP = json.dumps({'method': 'qwen3_next_mtp', 'num_speculative_tokens': 2})
SERVE_ARGS = f"--language-model-only --speculative-config '{MTP}'" if QWEN35 else ''
EXTRA_BODY = '{"chat_template_kwargs": {"enable_thinking": false}}' if QWEN35 else ''
DTYPE = 'bfloat16' if QWEN35 and compute_cap >= 8.0 else 'half'
if QWEN35 and compute_cap < 8.0:
    print('⚠️ Qwen3.5 on a pre-Ampere GPU runs in fp16, untested; see /content/vllm.log if vLLM fails')
# Batches in flight (--workers in section 4): about what fits in the KV cache
# at ~7k tokens per batch, so vLLM stays full without requests queueing past
# the 300 s client timeout
WORKERS = {'fast': 40, 'fastest': 56, 'qwen3.5-2b': 64, 'qwen3.5-0.8b': 64,
           'qwen3.5-4b': 8 if gpu_gb < 20 else 32}.get(LABEL_MODEL, 24)
print(f'{gpu_gb:.0f} GB GPU (compute {compute_cap}) -> {MODEL}, {DTYPE}, {WORKERS} workers'
      + (', MTP' if QWEN35 else ''))

# max-model-len: prompt (~6k tokens for 20 files) + max_tokens 8192 must fit
!nohup vllm serve {MODEL} --served-model-name {SERVED_NAME} --dtype {DTYPE} --max-model-len 16384 --gpu-memory-utilization 0.90 --port 8000 {SERVE_ARGS} > /content/vllm.log 2>&1 &

In [ ]:
# Wait for the server (first start downloads the weights: a few minutes)
import time, urllib.request
for _ in range(120):
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=5); print('vLLM ready'); break
    except Exception:
        time.sleep(10)
else:
    print('vLLM did not start — see the log below')
!tail -n 20 /content/vllm.log

In [ ]:
os.environ['LOCAL_LLM_BASE_URL'] = 'http://localhost:8000/v1'
os.environ['LOCAL_LLM_MODEL']    = SERVED_NAME
os.environ['LOCAL_LLM_API_KEY']  = 'not-needed'
os.environ['LOCAL_LLM_EXTRA_BODY'] = EXTRA_BODY   # '' except for Qwen3.5 (thinking off)

## 3. Check label quality on the gold set

Compare accuracy / precision / recall with the sonnet numbers before labeling everything.

In [ ]:
!test -f model/tmp/eval_llm_labels.py && python -m model.tmp.eval_llm_labels --source /content/source --repeat 3

## 4. Label

`--workers` keeps several batches in flight so vLLM can batch them. Re-running this cell after a disconnect
(re-run sections 1–2 first) only labels files missing from the cache.

In [ ]:
OUTPUT = f'{WORK}/llm_training_data_{SERVED_NAME}.jsonl'
!python -m model.generate_training_data --root /content/source --output {OUTPUT} --cache {WORK}/llm_label_cache.jsonl --workers {WORKERS}

In [ ]:
# Replies the parser had to salvage or gave up on
!cp model/tmp/llm_failed_replies.log {WORK}/ 2>/dev/null; tail -c 2000 model/tmp/llm_failed_replies.log 2>/dev/null || echo "no failed replies"

Download `llm_training_data_<model>.jsonl` from Drive and train locally (README, Step 2):

```bash
python -m model.train_ml_scorer train --data ./model/tmp/llm_training_data_<model>.jsonl --model llm_scorer_v3.pkl --sources llm,auto
```